In [11]:
import numpy as np
import cv2
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
%matplotlib inline

In [12]:
def get_full_path(annotations_path, filename):
    return os.path.join(annotations_path, filename)

def load_json(filename):
    with open(filename) as file:
        data_dict = json.load(file)
    return data_dict

In [7]:
def get_df_from_dict(json_dict, images_path):
    images_path = images_path
    json_dict = trainval_dict
    joint_coordinate_names = ['R_ANKLE_X', 'R_ANKLE_Y', 'R_KNEE_X', 'R_KNEE_Y', 'R_HIP_X',
           'R_HIP_Y', 'L_HIP_X', 'L_HIP_Y', 'L_KNEE_X', 'L_KNEE_Y', 'L_ANKLE_X',
           'L_ANKLE_Y', 'PELVIS_X', 'PELVIS_Y', 'THORAX_X', 'THORAX_Y',
           'UPPER_NECK_X', 'UPPER_NECK_Y', 'HEAD_TOP_X', 'HEAD_TOP_Y', 'R_WRIST_X',
           'R_WRIST_Y', 'R_ELBOW_X', 'R_ELBOW_Y', 'R_SHOULDER_X', 'R_SHOULDER_Y',
           'L_SHOULDER_X', 'R_SHOULDER_Y', 'L_ELBOW_X', 'L_ELBOW_Y', 'L_WRIST_X',
           'L_WRIST_Y']
    joint_names = joint_coordinate_names[::2]
    joint_names = [i[:-2] for i in joint_names]
    visibility_names = [i[:-2]+'_visibility' for i in joint_names]
    column_names = ['filename', 'center_X', 'center_Y', 'scale'] + joint_coordinate_names + visibility_names + ['height', 'width']

    df = pd.DataFrame(columns=column_names)

    # all training images
    for index, current_sample in tqdm(enumerate(json_dict), total=len(json_dict)):
        current_filename = current_sample['image']

        image_full_path = os.path.join(images_path, current_filename)
        image = cv2.imread(image_full_path, 1)
        image_shape = image.shape
        if len(image_shape) == 3:
            height, width, num_chanels = image_shape
        else:
            height, width = image_shape
            num_chanels = 1

        current_joints_vis = np.array(current_sample['joints_vis'])
        current_joints_coords = np.array(current_sample['joints']).flatten()
        current_joints_coords[::2] = current_joints_coords[::2] / width
        current_joints_coords[1::2] = current_joints_coords[1::2] / height

        current_scale = np.array(current_sample['scale']).reshape(1,)
        current_center = np.array(current_sample['center']).flatten()
        current_center[0] /= width
        current_center[1] /= height
        height_width = np.array([height, width])

        numeric_values = np.concatenate((current_center, current_scale, current_joints_coords, current_joints_vis, height_width))

        current_row = list([current_filename]) + list(numeric_values)
        df = df.append(pd.Series(current_row, index=column_names), ignore_index=True)
    return df

In [8]:
dataset_path = os.path.join('data', 'mpii')
annotations_path = os.path.join(dataset_path, 'annotations')
images_path = os.path.join(dataset_path, 'images')

valid_matfile, mpii_annotations_2_file, test_file, train_file, trainval_file = os.listdir(annotations_path)

valid_matfile = get_full_path(annotations_path, valid_matfile)
mpii_annotations_2_file = get_full_path(annotations_path, mpii_annotations_2_file)
test_file = get_full_path(annotations_path, test_file)
train_file = get_full_path(annotations_path, train_file)
trainval_file = get_full_path(annotations_path, trainval_file)

train_dict = load_json(train_file)
trainval_dict = load_json(trainval_file)
test_dict = load_json(test_file)

keys = list(train_dict[0].keys())

In [9]:
df_trainval = get_df_from_dict(trainval_dict, images_path)

# images containig only one human
df_trainval_unique = df_trainval.drop_duplicates(subset=['filename'], keep=False)
df_trainval_unique.reset_index(drop=True, inplace=True)

# save both df
df_trainval.to_csv('trainval.csv', index=False)
df_trainval_unique.to_csv('trainval_unique.csv', index=False)

100%|████████████████████████████████████████████████████████████████████████████| 29116/29116 [22:49<00:00, 21.25it/s]


In [93]:
images_path = images_path
json_dict = trainval_dict
joint_coordinate_names = ['R_ANKLE_X', 'R_ANKLE_Y', 'R_KNEE_X', 'R_KNEE_Y', 'R_HIP_X',
       'R_HIP_Y', 'L_HIP_X', 'L_HIP_Y', 'L_KNEE_X', 'L_KNEE_Y', 'L_ANKLE_X',
       'L_ANKLE_Y', 'PELVIS_X', 'PELVIS_Y', 'THORAX_X', 'THORAX_Y',
       'UPPER_NECK_X', 'UPPER_NECK_Y', 'HEAD_TOP_X', 'HEAD_TOP_Y', 'R_WRIST_X',
       'R_WRIST_Y', 'R_ELBOW_X', 'R_ELBOW_Y', 'R_SHOULDER_X', 'R_SHOULDER_Y',
       'L_SHOULDER_X', 'R_SHOULDER_Y', 'L_ELBOW_X', 'L_ELBOW_Y', 'L_WRIST_X',
       'L_WRIST_Y']
joint_names = joint_coordinate_names[::2]
joint_names = [i[:-2] for i in joint_names]
visibility_names = [i[:-2]+'_visibility' for i in joint_names]
column_names = ['filename', 'center_X', 'center_Y', 'scale'] + joint_coordinate_names + visibility_names + ['height', 'width']

df = pd.DataFrame(columns=column_names)

# all training images
for index, current_sample in tqdm(enumerate(json_dict), total=len(json_dict)):
    current_filename = current_sample['image']
    
    image_full_path = os.path.join(images_path, current_filename)
    image = cv2.imread(image_full_path, 1)
    image_shape = image.shape
    if len(image_shape) == 3:
        height, width, num_chanels = image_shape
    else:
        height, width = image_shape
        num_chanels = 1
    
    current_joints_vis = np.array(current_sample['joints_vis'])
    current_joints_coords = np.array(current_sample['joints']).flatten()
    current_joints_coords[::2] = current_joints_coords[::2] / width
    current_joints_coords[1::2] = current_joints_coords[1::2] / height
    
    current_scale = np.array(current_sample['scale']).reshape(1,)
    current_center = np.array(current_sample['center']).flatten()
    current_center[0] /= width
    current_center[1] /= height
    height_width = np.array([height, width])
    
    numeric_values = np.concatenate((current_center, current_scale, current_joints_coords, current_joints_vis, height_width))

    current_row = list([current_filename]) + list(numeric_values)
    df = df.append(pd.Series(current_row, index=column_names), ignore_index=True)
    break

  0%|          | 0/29116 [00:00<?, ?it/s]


In [94]:
df

,filename,center_X,center_Y,scale,R_ANKLE_X,R_ANKLE_Y,R_KNEE_X,R_KNEE_Y,R_HIP_X,R_HIP_Y,...,UPPER_NE_visibility,HEAD_T_visibility,R_WRI_visibility,R_ELB_visibility,R_SHOULD_visibility,L_SHOULD_visibility,L_ELB_visibility,L_WRI_visibility,height,width
0,015601864.jpg,0.464062,0.356944,3.021046,0.484375,0.547222,0.48125,0.373611,0.447656,0.256944,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,720.0,1280.0


In [10]:
column_names

NameError: name 'column_names' is not defined